In [4]:
import psycopg2

# connect เข้า postgres หลักก่อน
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="672437002",
    host="localhost",
    port="5432"
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("CREATE DATABASE api_ptt;")
print("Database api_ptt created")

cur.close()
conn.close()


Database api_ptt created


In [6]:
conn = psycopg2.connect(
    dbname="api_ptt",
    user="postgres",
    password="672437002",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
CREATE TABLE stocks (
    stock_id SERIAL PRIMARY KEY,
    symbol VARCHAR(10) UNIQUE NOT NULL,
    instrument_type VARCHAR(20),
    security_type VARCHAR(10),
    exchange VARCHAR(20)
);

CREATE TABLE market_status (
    status_id SERIAL PRIMARY KEY,
    market_status VARCHAR(20),
    security_status VARCHAR(10)
);

CREATE TABLE stock_quotes (
    quote_id SERIAL PRIMARY KEY,
    stock_id INT REFERENCES stocks(stock_id),
    status_id INT REFERENCES market_status(status_id),
    last_price NUMERIC,
    high_price NUMERIC,
    low_price NUMERIC,
    pe NUMERIC,
    pbv NUMERIC,
    eps NUMERIC,
    total_volume BIGINT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")

conn.commit()
print("Tables created successfully")

cur.close()
conn.close()


Tables created successfully


In [8]:
import psycopg2

conn = psycopg2.connect(
    dbname="api_ptt",
    user="postgres",
    password="672437002",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
""")

cur.fetchall()


[('stocks',), ('stock_quotes',), ('market_status',)]

In [10]:
from settrade_v2 import Investor

investor = Investor(
    app_id="7j05EwuFPvcBsYfb",
    app_secret="AJUDHizWtI9Kd/LG5GEOwPzvHQmxaD9HLBN9WxS3S5ef",
    broker_id="SANDBOX",
    app_code="SANDBOX"
)

mkt_data = investor.MarketData()
ptt = mkt_data.get_quote_symbol("PTT")

ptt


{'instrumentType': 'STOCK',
 'symbol': 'PTT',
 'high': 31.25,
 'low': 31.25,
 'last': 31.25,
 'average': 31.25,
 'change': 7.15,
 'percentChange': 29.67,
 'totalVolume': 1200,
 'totalBuyVolume': 0,
 'totalSellVolume': 0,
 'totalNoSideVolume': 1200,
 'status': '',
 'marketStatus': 'Close',
 'securityType': 'CS',
 'eps': 2.26,
 'pe': 13.31,
 'pbv': 0.88,
 'percentYield': 6.09,
 'maturityDate': None,
 'exercisePrice': None,
 'underlying': None,
 'underlyingPrice': None,
 'intrinsicValue': None,
 'theoretical': None,
 'moneyness': None,
 'lastTradingDate': None,
 'toLastTrade': None,
 'exerciseRatio': None,
 'impliedVolatility': None,
 'exchange': None,
 'aumSize': None,
 'inav': None}

In [18]:
cur.execute("""
INSERT INTO stocks (symbol, instrument_type, security_type, exchange)
VALUES (%s,%s,%s,%s)
ON CONFLICT (symbol) DO NOTHING
""", (
    ptt["symbol"],
    ptt["instrumentType"],
    ptt["securityType"],
    ptt["exchange"]
))


In [20]:
cur.execute("""
INSERT INTO market_status (market_status, security_status)
VALUES (%s,%s)
RETURNING status_id
""", (
    ptt["marketStatus"],
    ptt["status"]
))
status_id = cur.fetchone()[0]


In [22]:
cur.execute("SELECT stock_id FROM stocks WHERE symbol='PTT'")
stock_id = cur.fetchone()[0]

cur.execute("""
INSERT INTO stock_quotes
(stock_id, status_id, last_price, high_price, low_price,
 pe, pbv, eps, total_volume)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
""", (
    stock_id,
    status_id,
    ptt["last"],
    ptt["high"],
    ptt["low"],
    ptt["pe"],
    ptt["pbv"],
    ptt["eps"],
    ptt["totalVolume"]
))

conn.commit()


In [24]:
cur.execute("""
SELECT s.symbol,
       q.last_price,
       q.pe,
       q.pbv,
       q.created_at
FROM stock_quotes q
JOIN stocks s ON q.stock_id = s.stock_id
ORDER BY q.created_at DESC
LIMIT 5
""")

cur.fetchall()


[('PTT',
  Decimal('31.25'),
  Decimal('13.31'),
  Decimal('0.88'),
  datetime.datetime(2026, 2, 5, 16, 15, 55, 725464))]

In [ ]:
import time

while True:
    # ดึง PTT
    # insert
    time.sleep(86400)


In [ ]:
import time
import psycopg2
from settrade_v2 import Investor


In [ ]:
investor = Investor(
    app_id="7j05EwuFPvcBsYfb",
    app_secret="AJUDHizWtI9Kd/LG5GEOwPzvHQmxaD9HLBN9WxS3S5ef",
    broker_id="SANDBOX",
    app_code="SANDBOX"
)

mkt_data = investor.MarketData()


In [ ]:
conn = psycopg2.connect(
    dbname="api_ptt",
    user="postgres",
    password="672437002",
    host="localhost",
    port="5432"
)
cur = conn.cursor()


In [ ]:
while True:
    print("Fetching PTT data...")

    # 1. ดึงข้อมูล PTT
    ptt = mkt_data.get_quote_symbol("PTT")

    # 2. insert stock (ถ้ายังไม่มี)
    cur.execute("""
    INSERT INTO stocks (symbol, instrument_type, security_type, exchange)
    VALUES (%s,%s,%s,%s)
    ON CONFLICT (symbol) DO NOTHING
    """, (
        ptt["symbol"],
        ptt["instrumentType"],
        ptt["securityType"],
        ptt["exchange"]
    ))

    # 3. insert market status
    cur.execute("""
    INSERT INTO market_status (market_status, security_status)
    VALUES (%s,%s)
    RETURNING status_id
    """, (
        ptt["marketStatus"],
        ptt["status"]
    ))
    status_id = cur.fetchone()[0]

    # 4. หา stock_id
    cur.execute("SELECT stock_id FROM stocks WHERE symbol='PTT'")
    stock_id = cur.fetchone()[0]

    # 5. insert quote (time-series)
    cur.execute("""
    INSERT INTO stock_quotes
    (stock_id, status_id, last_price, high_price, low_price,
     pe, pbv, eps, total_volume)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (
        stock_id,
        status_id,
        ptt["last"],
        ptt["high"],
        ptt["low"],
        ptt["pe"],
        ptt["pbv"],
        ptt["eps"],
        ptt["totalVolume"]
    ))

    conn.commit()

    print("Inserted PTT data at", time.strftime("%Y-%m-%d %H:%M:%S"))

    time.sleep(10)


In [ ]:
cur.execute("""
SELECT s.symbol,
       q.last_price,
       q.pe,
       q.pbv,
       q.created_at
FROM stock_quotes q
JOIN stocks s ON q.stock_id = s.stock_id
ORDER BY q.created_at DESC
LIMIT 5
""")

cur.fetchall()
